In [ ]:
%load_ext autoreload
%autoreload 2

### First, this notebook creates a static version of the MODIS product to classify MTBS fires outside of the 2003-2016 MODIS data range. Then, it classifies all fires in MTBS. Finally, this notebook runs a time-varying fit on MTBS data (in 2 grouping methods: 1. individual land cover types and 2. some land cover types split geographically by clusters and others grouped together) in the USA and creates visualizations to show how the CCDF shape changes over time.

In [ ]:
import geopandas as gpd

from wildfire_stats import config as cfg
from wildfire_stats.data import classification as dcls
from wildfire_stats.data import io as dio
from wildfire_stats.data import rasters as drast
from wildfire_stats.fitting import fits as fits
from wildfire_stats.plotting import distributions as pdist
from wildfire_stats.plotting import landcover as plc
from wildfire_stats.plotting import temporal as ptemp
from wildfire_stats.reporting import summaries as rsum

P = cfg.load_paths()
A = cfg.load_assumptions()

First, build the static raster tiles (so that it can be applied to fires outside 2003-2016)

In [ ]:
mtbs_gdf = dio.load_mtbs(P.mtbs / "S_USA.MTBS_BURN_AREA_BOUNDARY.shp")

mtbs_filtered = mtbs_gdf[mtbs_gdf["AREA_SQKM"] >= A["min_fire"]].copy()
mtbs_filtered = mtbs_filtered.rename(columns={"YEAR": "year", "AREA_SQKM": "area_km2"})

In [ ]:
# SLOW: writes the static MODIS tiles to P.logged_output / "static_modis_tiles"
# (GeoTIFFs created in notebook 3)
modis_by_year = drast.index_modis_by_year(P.modis / "GeoTIFF")

static_raster = drast.build_static_modis_tiles(modis_by_year)

Use the static raster to classify all MTBS fires.

In [ ]:
# SLOW: after the first run, skip this and the previous cell and reload instead:
# mtbs_classified = dio.restore_truncated_columns(gpd.read_file(P.logged_output / "mtbs_classified_static.shp"))

mtbs_classified = dcls.classify_with_modis(mtbs_filtered, "year", modis_by_year)
mtbs_classified = dcls.classify_with_static(mtbs_classified, static_raster)

dio.save_vector(mtbs_classified, P.logged_output / "mtbs_classified_static.shp")

Evaluate how well new static raster mapping matches original time-varying raster.

In [ ]:
agreement_rate, comparison_df = rsum.classification_agreement(
    mtbs_classified, "modis_class_timevary", "modis_class_static", years=A["modis_years"]
)
print(comparison_df[["year", "area_km2", "modis_class_timevary", "modis_class_static", "match"]].head())

## 2 visualizations to justify keeping xmin = 4 cutoff...

In [ ]:
ptemp.plot_fire_counts_faceted(mtbs_classified)

To visualize where fires in each grouping occur:

In [ ]:
plc.plot_biome_facets(mtbs_classified)

In [ ]:
counts_long, counts_wide = rsum.fires_time_series_by_ecoregion(
    mtbs_classified,
    year_col="year",
    class_col="modis_class_static",
    period_years=4,
    include_total=False,   # optional
    plot=True
)

In [ ]:
ptemp.fire_threshold_analysis(mtbs_classified,
                              year_col="year",
                              area_col="area_km2",
                              thresholds=[4, 8, 12, 16, 20])

Thus, there is no other min fire size threshold that will eliminate large jumps in fire size after a certain year, meaning we can stick with 4km2. Now, let's run the fitting routine on each biome using the new land cover classification.

In [ ]:
# SLOW
overall_results = fits.fit_static_by_group(mtbs_classified, "modis_class_static")

dio.save_pickle(overall_results, P.output / "mtbs_static_fits_by_biome.pkl")

And now, we run the time-varying version of the fitting routine where each parameter is written as p1+p2*t (with modification for truncated power law, see utils). 

In [ ]:
timevary_results = fits.analyze_time_varying_mle(mtbs_classified, overall_results)

dio.save_pickle(timevary_results, P.output / "mtbs_time_varying_fits_by_biome.pkl")

Now, to view the results from the time-varying fits...

In [ ]:
df_both, df_p1, df_p2 = rsum.summarize_timevary_modes(timevary_results)
df_both

Or just the "main" ones...

In [ ]:
summary_table = rsum.summarize_fits(df_both, df_p1, df_p2, overall_results, include="primary")
summary_table

And just the additional fits...

In [ ]:
summary_other = rsum.summarize_fits(df_both, df_p1, df_p2, overall_results, include="other")
summary_other

Or all of them in that format...

In [ ]:
results_table = rsum.summarize_fits(df_both, df_p1, df_p2, overall_results, include="all", order="primary_first")
results_table

Below are a set of plotting functions to visualize how well the fits match the data qualitatively.

In [ ]:
pdist.plot_group_ccdf_cdf_qq(
    mtbs_classified, "Mixed forest", col_name="modis_class_static",
    which=["generalized_pareto", "power_law"], nonstationary=False,
)

In [ ]:
ptemp.plot_distribution_evolution_ccdf_pdf(
    df_both,
    mtbs_classified,
    df_p1=df_p1,
    df_p2=df_p2,
)

The following function runs a Poisson regression on each land cover type and is used to determine if change in tail shape is due to relative change in either small or large fires (or both).

In [ ]:
poisson_by_biome = fits.fit_poisson_tail_trend(mtbs_classified, valid_groups=dcls.POISSON_BIOMES)
poisson_by_biome

Based on the spatial distribution of fires in each land cover type, the following cell groups them into the groupings and reruns the fitting routine on these new groupings.

- Deciduous Broadleaf forests
- Evergreen Broadleaf forests
- Evergreen Broadleaf forests in the American South
- Evergreen Broadleaf forests on the West Coast
- Grasslands
- Grasslands and American Southwest Open Shrublands
- Open Shrublands
- Open Shrublands in the American Southwest
- Open Shrublands in Alaska

The other biomes are not included in this grouping because they were too scattered around the US without clear clusters.

In [ ]:
# Nine overlapping groupings (base biomes / longitude splits from default_assumptions.yaml)
df_refined = dcls.build_refined_groups(mtbs_classified)

overall_results_refined = fits.fit_static_by_group(df_refined, "biome_refined")

dio.save_pickle(overall_results_refined, P.output / "mtbs_static_fits_by_biome_refined.pkl")

And then we run the time-varying fits on these groupings.

In [ ]:
timevary_results_refined = fits.analyze_time_varying_mle(
    df_refined, overall_results_refined, group_col="biome_refined"
)
df_both_refined, df_p1_refined, df_p2_refined = rsum.summarize_timevary_modes(timevary_results_refined)

In [ ]:
timevary_summary = rsum.summarize_fits(df_both_refined, df_p1_refined, df_p2_refined, overall_results_refined)
timevary_summary

In [ ]:
fits.fit_poisson_tail_trend(
    df_refined, group_col="biome_refined", label_col="biome_refined",
    title="Poisson Trend in Fire Frequency vs. Size Threshold (Refined Biomes)",
)
ptemp.plot_distribution_evolution_ccdf_pdf(
    df_both_refined, df_refined, df_p1_refined, df_p2_refined,
    group_col="biome_refined",
    xmin=12,  # TODO(bug): the original call relied on the old default xmin=12, not min_fire
)

Now, the same plotting visualizations on these refined groupings. This one now includes a 4th panel where the theoretical quantiles take into account the year of the fire and the (interpolated) distribution parameters in that year (from timevary_summary).

In [ ]:
fig = pdist.plot_group_ccdf_cdf_qq(
    df_refined,
    "Open shrublands W of 130W",
    col_name="biome_refined",
    which=("lognormal",),
    year_col="year",
    timevary_summary=timevary_summary,
    timevary_dist="lognormal",
)

Next, we want to check: are we not seeing power law behavior because we are looking at too small of fires (as opposed to just the tail)? So, the following function incrementally increases the minimum fire size and refits the data for a given grouping. It returns the minimum fire size where a power law is a "good" fit for fires above that size (if one exists). It then returns visualizations showing how that power law fit looks on the entire dataset (left panel) and just the tail (right panel).

In [ ]:
fig, tail_info = pdist.plot_tail_scan_ccdf_cdf_qq(
    df_refined,
    "Open shrublands E of 130W",
    col_name="biome_refined",
    which=("lognormal",),  # or whatever you’ve been using
    candidate_percentiles=[50, 60, 70, 80, 85, 90, 92, 94, 96, 97, 98],
    min_tail_n=100,
    R_boot_tail=20,
)

In [ ]:
# TODO(bug): df_refined has overlapping (duplicated) groups and only 4 base biomes,
# yet notebook 7 reads this file as the MTBS fire table (HANDOFF 4b).
dio.save_vector(df_refined, P.logged_output / "mtbs_classified.shp")